In [ ]:
from netCDF4 import Dataset

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

import numpy as np
import numpy.ma as ma
import time
from  datetime import date,timedelta
from dateutil.relativedelta import relativedelta
import statsmodels.api as sm

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.cm as cm
import matplotlib.path as mpath
from matplotlib.colors import ListedColormap

from scipy.optimize import curve_fit
import pandas as pd
#import nctoolkit as nc
import xarray as xr

In [ ]:
dirpath = '../data/CRU/grid/'
ncfile = Dataset(dirpath+'cru_ts4.06.1901.2021.tmp.dat.nc')
print(ncfile)

In [ ]:
lat = np.squeeze(ncfile.variables['lat'])
lat = lat[300::]

lon = np.squeeze(ncfile.variables['lon'])

tmp = np.squeeze(ncfile.variables['tmp'])
tmp = tmp[1188::,300::,:]
print(tmp.shape)

In [ ]:
time  = np.squeeze(ncfile.variables["time"])
#time = np.array([date(2000,1,1)  + relativedelta(months=x)for x in time])
time = np.array([date(2000,1,1) + timedelta(days=int(x)) for x in time])
year=np.array([x.year for x in time])
month=np.array([x.month for x in time])

In [ ]:
## y=ax+b를 리턴합니다.
def func(x, a, b):
    return a*x + b

In [ ]:
ave = np.empty((12,60,720))
std = np.empty((12,60,720))
trend = np.empty((12,60,720))
trend_var = np.empty((12,60,720))
b = np.empty((12,60,720))
b_var = np.empty((12,60,720))

ave[:] = np.nan
std[:] = np.nan
trend[:] = np.nan
trend_var[:] = np.nan
b[:]= np.nan
b_var[:]= np.nan

for x in range(len(lon)):
    for y in range(len(lat)):
        ii=[]
        li=[]
        for i in range(0,2):
            ii = list(range(12+i,264,12))
            time =np.arange(2001,2022,1)
            for ll in range(len(ii)):
                a = tmp[ii[ll],y,x]
                if a > 300:
                    a=np.nan
                li=np.append(li, a)
            ave[i,y,x] =  np.mean(li)
            std[i,y,x] =  np.std(li)
            if np.isnan(li).any():
                a
            else:                
                popt, pcov = curve_fit(func, time, li)
                trend[i,y,x] = popt[0]
                trend_var[i,y,x] = pcov[0][0]
                b[i,y,x] = popt[1]
                b_var[i,y,x] = pcov[1][1]
            
            li=[]
            ii=[]
        for i in range(2,12):
            ii = list(range(0+i,264,12))
            time =np.arange(2000,2022,1)
            for ll in range(len(ii)):
                a = tmp[ii[ll],y,x]
                if a > 300:
                    a=np.nan
                li=np.append(li, a)
            ave[i,y,x] =  np.mean(li)
            std[i,y,x] =  np.std(li)
            if np.isnan(li).any():
                a
            else:                
                popt, pcov = curve_fit(func, time, li)
                trend[i,y,x] = popt[0]
                trend_var[i,y,x] = pcov[0][0]
                b[i,y,x] = popt[1]
                b_var[i,y,x] = pcov[1][1]

            li=[]
            ii=[]
            
np.save(dirpath+'ave.npy', ave)           
np.save(dirpath+'std.npy', std)           
np.save(dirpath+'trend.npy', trend)
np.save(dirpath+'trend_var.npy', trend_var)
np.save(dirpath+'b.npy', b)
np.save(dirpath+'b_var.npy', b_var)

In [ ]:
all_m = np.empty((12,4))
all_t = np.empty((12,4))
all_b = np.empty((12,4))
all_s = np.empty((12,4))

all_m[:,:]= np.nan
all_t[:,:]= np.nan
all_b[:,:]= np.nan
all_s[:,:]= np.nan

ave1 = ave.copy()
ave1[:,lat>70,:]=np.nan
ave1[:,:,lon<0]=np.nan
ave2 = ave.copy()
ave2[:,lat>70,:]=np.nan
ave2[:,:,lon<-160]=np.nan
ave2[:,:,lon>-90]=np.nan
ave3 = ave.copy()
ave3[:,lat>80,:]=np.nan
ave3[:,:,lon<-60]=np.nan
ave3[:,:,lon>-20]=np.nan

tr1 = trend.copy()
tr1[:,lat>70,:]=np.nan
tr1[:,:,lon<0]=np.nan
tr2 = trend.copy()
tr2[:,lat>70,:]=np.nan
tr2[:,:,lon<-160]=np.nan
tr2[:,:,lon>-90]=np.nan
tr3 = trend.copy()
tr3[:,lat>80,:]=np.nan
tr3[:,:,lon<-60]=np.nan
tr3[:,:,lon>-20]=np.nan


b1 = b.copy()
b1[:,lat>70,:]=np.nan
b1[:,:,lon<0]=np.nan
b2 = b.copy()
b2[:,lat>70,:]=np.nan
b2[:,:,lon<-160]=np.nan
b2[:,:,lon>-90]=np.nan
b3 = b.copy()
b3[:,lat>80,:]=np.nan
b3[:,:,lon<-60]=np.nan
b3[:,:,lon>-20]=np.nan

s1 = std.copy()
s1[:,lat>70,:]=np.nan
s1[:,:,lon<0]=np.nan
s2 = std.copy()
s2[:,lat>70,:]=np.nan
s2[:,:,lon<-160]=np.nan
s2[:,:,lon>-90]=np.nan
s3 = std.copy()
s3[:,lat>80,:]=np.nan
s3[:,:,lon<-60]=np.nan
s3[:,:,lon>-20]=np.nan

for i in range(0,12):
    all_m[i,0] = np.nanmean(ave[i,:,:])
    all_m[i,1] = np.nanmean(ave1[i,:,:])
    all_m[i,2] = np.nanmean(ave2[i,:,:])
    all_m[i,3] = np.nanmean(ave3[i,:,:])

    all_s[i,0] = np.nanmean(std[i,:,:])
    all_s[i,1] = np.nanmean(s1[i,:,:])
    all_s[i,2] = np.nanmean(s2[i,:,:])
    all_s[i,3] = np.nanmean(s3[i,:,:])
   
    all_b[i,0] = np.nanmean(b[i,:,:])
    all_b[i,1] = np.nanmean(b1[i,:,:])
    all_b[i,2] = np.nanmean(b2[i,:,:])
    all_b[i,3] = np.nanmean(b3[i,:,:])

    all_t[i,0] = np.nanmean(trend[i,:,:]*10.)
    all_t[i,1] = np.nanmean(tr1[i,:,:]*10.)
    all_t[i,2] = np.nanmean(tr2[i,:,:]*10.)
    all_t[i,3] = np.nanmean(tr3[i,:,:]*10.)
                               
df=pd.DataFrame(all_m)
print(df)
df.to_csv(dirpath + 'all_md_m.csv', mode='w',index=False, header=False)
df=pd.DataFrame(all_t)
print(df)
df.to_csv(dirpath + 'all_md_t.csv', mode='w',index=False, header=False)
df=pd.DataFrame(all_b)
print(df)
df.to_csv(dirpath + 'all_md_b.csv', mode='w',index=False, header=False)
df=pd.DataFrame(all_s)
print(df)
df.to_csv(dirpath + 'all_md_s.csv', mode='w',index=False, header=False)

In [ ]:
cenLon = 0
minLat = 55
maxLat = 90
textLat = minLat - 15
textLon = cenLon
month = ['Jan', 'Feb', 'Mar', 'Apr','May','Jun','Jul','Aug','Sep', 'Oct', 'Nov', 'Dec']

In [ ]:
#Color
contour_levels = np.arange(-5,5.2,0.2)
bwr = cm.get_cmap('bwr', 100)
bwr = bwr(np.arange(100))
#white = np.array([1.,1.,1.,1])
#jet[70:85,:]= white
new_cmap = ListedColormap(bwr)

In [ ]:
stmm = 1
enmm = 1

for mm in np.arange(stmm,enmm+1):

    fig = plt.figure(figsize=(6,6))
    ax = plt.axes(projection=ccrs.NorthPolarStereo(central_longitude = cenLon))
    ax.set_extent([0, 359, 60, 90], crs=ccrs.PlateCarree())

    ax.gridlines(color='gray', linestyle='--',linewidth=0.5)

    # 원형테두리
    theta = np.linspace(0,2*np.pi,144)
    center = [0.5,0.5]
    radius = 0.5
    points = np.array([np.cos(theta),np.sin(theta)]).T
    circle = mpath.Path(points*radius+center)
    ax.set_boundary(circle,transform=ax.transAxes)

    #shading
    image = ax.contourf(lon,lat,trend[mm-1,:,:]*10.,contour_levels, cmap=new_cmap, transform=ccrs.PlateCarree(), extend='both')
    plt.gca().coastlines()
    ax.text(0, 80+0.7, ' 80°N', fontsize = 8.2, ha ='center', color='gray',transform = ccrs.PlateCarree())
    ax.text(0, 70+1.0, ' 70°N', fontsize = 8.2, ha ='center', color='gray',transform = ccrs.PlateCarree())
    ax.text(0, 60+1.3, ' 60°N', fontsize = 8.2, ha ='center', color='gray',transform = ccrs.PlateCarree())
    ax.text(60, 58., ' 60°E', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(120, 58, ' 120°E', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(180, 59.7, ' 180°', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(-60, 57.8, '60°W  ', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(-120, 58.3, '120°W  ', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(0, 58., ' 0°', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())

    #colorbar
#    caxes = fig.add_axes([0.1,0.07,0.86,0.03])
#    cbar =plt.colorbar(image, orientation = 'horizontal', ticks = contour_levels[::10],drawedges = True, cax=caxes)
#    cbar.ax.tick_params(direction='in')
    #cbar.ax.set_xticklables([int(x) for x in contour_levels[::2]])

    #title
#    plt.suptitle('Spatial Interpolation change rate (' +str(month[mm-1]) +')', fontsize = 15, y=0.95)
    #ax.text(textLon, textLat, 'explanation', fontsize = 8, ha ='center', transform = ccrs.Geodetic())

 #    plt.show()
    plt.savefig(dirpath+'trend'+ str(mm) + '_gr.png', dpi=300, trnsparent=True)
    plt.savefig(dirpath+'trend'+ str(mm) + '_gr.eps')

In [ ]:
#Color
contour_levels = np.arange(-56,24,4)
jet = cm.get_cmap('bwr', 100)
jet = jet(np.arange(100))
#white = np.array([1.,1.,1.,1])
#jet[70:85,:]= white
jet = jet[0:68]
new_cmap = ListedColormap(jet)

In [ ]:
stmm = 1
enmm = 12

for mm in np.arange(stmm,enmm+1):

    fig = plt.figure(figsize=(6,6))
    ax = plt.axes(projection=ccrs.NorthPolarStereo(central_longitude = cenLon))
    ax.set_extent([0, 359, 60, 90], crs=ccrs.PlateCarree())

    ax.gridlines(color='gray', linestyle='--',linewidth=0.5)

    # 원형테두리
    theta = np.linspace(0,2*np.pi,144)
    center = [0.5,0.5]
    radius = 0.5
    points = np.array([np.cos(theta),np.sin(theta)]).T
    circle = mpath.Path(points*radius+center)
    ax.set_boundary(circle,transform=ax.transAxes)

    #shading
    image = ax.contourf(lon,lat,ave[mm-1,:,:],contour_levels, cmap=new_cmap, transform=ccrs.PlateCarree(), extend='both')
    plt.gca().coastlines()
    ax.text(0, 80+0.7, ' 80°N', fontsize = 8.2, ha ='center', color='gray',transform = ccrs.PlateCarree())
    ax.text(0, 70+1.0, ' 70°N', fontsize = 8.2, ha ='center', color='gray',transform = ccrs.PlateCarree())
    ax.text(0, 60+1.3, ' 60°N', fontsize = 8.2, ha ='center', color='gray',transform = ccrs.PlateCarree())
    ax.text(60, 58., ' 60°E', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(120, 58, ' 120°E', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(180, 59.7, ' 180°', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(-60, 57.8, '60°W  ', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(-120, 58.3, '120°W  ', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(0, 58., ' 0°', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())

    #colorbar
#    caxes = fig.add_axes([0.1,0.07,0.86,0.03])
#    cbar =plt.colorbar(image, orientation = 'horizontal', ticks = contour_levels[::2],drawedges = True, cax=caxes)
#    cbar.ax.tick_params(direction='in')
    #cbar.ax.set_xticklables([int(x) for x in contour_levels[::2]])

    #title
#    plt.suptitle('Spatial Interpolation (' +str(month[mm-1]) +')', fontsize = 15, y=0.95)
    #ax.text(textLon, textLat, 'explanation', fontsize = 8, ha ='center', transform = ccrs.Geodetic())

 #    plt.show()
    plt.savefig(dirpath+'mean'+ str(mm) + '_gr.png', dpi=300, trnsparent=True)
    plt.savefig(dirpath+'mean'+ str(mm) + '_gr.eps')

# NO Greenland

In [ ]:
#Greenland 
tmp = np.squeeze(ncfile.variables['tmp'])
tmp = tmp[1188::,300::,:]

tmp1 = tmp
tmp1[:,:,((-60<=lon) & (lon<=-30))]=np.nan

a = np.asarray(np.where((-30<=lon) & (lon<=0))).flatten()
b = np.asarray(np.where(67.5<=lat)).flatten()
tmp1[:,b[0]:b[-1],a[0]:a[-1]]=np.nan

a = np.asarray(np.where((-60.8<=lon) & (lon<=-59.5))).flatten()
b = np.asarray(np.where((75<=lat) & (lat<=82.1))).flatten()
tmp1[:,b[0]:b[-1],a[0]:a[-1]]=np.nan

a = np.asarray(np.where((-61.8<=lon) & (lon<=-60.5))).flatten()
b = np.asarray(np.where((75<=lat) & (lat<=82))).flatten()
tmp1[:,b[0]:b[-1],a[0]:a[-1]]=np.nan

a = np.asarray(np.where((-62.8<=lon) & (lon<=-61.5))).flatten()
b = np.asarray(np.where((75<=lat) & (lat<=81.5))).flatten()
tmp1[:,b[0]:b[-1],a[0]:a[-1]]=np.nan

a = np.asarray(np.where((-65.3<=lon) & (lon<=-62.5))).flatten()
b = np.asarray(np.where((75<=lat) & (lat<=81.2))).flatten()
tmp1[:,b[0]:b[-1],a[0]:a[-1]]=np.nan

a = np.asarray(np.where((-65.8<=lon) & (lon<=-65))).flatten()
b = np.asarray(np.where((75<lat) & (lat<=81))).flatten()
tmp1[:,b[0]:b[-1],a[0]:a[-1]]=np.nan

a = np.asarray(np.where((-67.8<=lon) & (lon<=-65.5))).flatten()
b = np.asarray(np.where((75<=lat) & (lat<=80.7))).flatten()
tmp1[:,b[0]:b[-1],a[0]:a[-1]]=np.nan

a = np.asarray(np.where((-70.3<=lon) & (lon<=-67.5))).flatten()
b = np.asarray(np.where((75<=lat) & (lat<=80))).flatten()
tmp1[:,b[0]:b[-1],a[0]:a[-1]]=np.nan

a = np.asarray(np.where((-73.5<=lon) & (lon<=-70))).flatten()
b = np.asarray(np.where((75<=lat) & (lat<=79))).flatten()
tmp1[:,b[0]:b[-1],a[0]:a[-1]]=np.nan

In [ ]:
ave = np.empty((12,60,720))
std = np.empty((12,60,720))
trend = np.empty((12,60,720))
trend_var = np.empty((12,60,720))
b = np.empty((12,60,720))
b_var = np.empty((12,60,720))

ave[:] = np.nan
std[:] = np.nan
trend[:] = np.nan
trend_var[:] = np.nan
b[:]= np.nan
b_var[:]= np.nan

for x in range(len(lon)):
    for y in range(len(lat)):
        ii=[]
        li=[]
        for i in range(0,2):
            ii = list(range(12+i,264,12))
            time =np.arange(2001,2022,1)
            for ll in range(len(ii)):
                a = tmp1[ii[ll],y,x]
                if a > 300:
                    a=np.nan
                li=np.append(li, a)
            ave[i,y,x] =  np.mean(li)
            std[i,y,x] =  np.std(li)
            if np.isnan(li).any():
                a
            else:                
                popt, pcov = curve_fit(func, time, li)
                trend[i,y,x] = popt[0]
                trend_var[i,y,x] = pcov[0][0]
                b[i,y,x] = popt[1]
                b_var[i,y,x] = pcov[1][1]
            
            li=[]
            ii=[]
        for i in range(2,12):
            ii = list(range(0+i,264,12))
            time =np.arange(2000,2022,1)
            for ll in range(len(ii)):
                a = tmp1[ii[ll],y,x]
                if a > 300:
                    a=np.nan
                li=np.append(li, a)
            ave[i,y,x] =  np.mean(li)
            std[i,y,x] =  np.std(li)
            if np.isnan(li).any():
                a
            else:                
                popt, pcov = curve_fit(func, time, li)
                trend[i,y,x] = popt[0]
                trend_var[i,y,x] = pcov[0][0]
                b[i,y,x] = popt[1]
                b_var[i,y,x] = pcov[1][1]

            li=[]
            ii=[]
            
np.save(dirpath+'ave_gr.npy', ave)           
np.save(dirpath+'std_gr.npy', std)           
np.save(dirpath+'trend_gr.npy', trend)
np.save(dirpath+'trend_var_gr.npy', trend_var)
np.save(dirpath+'b_gr.npy', b)
np.save(dirpath+'b_var_gr.npy', b_var)

In [ ]:
all_m = np.empty((12,4))
all_t = np.empty((12,4))
all_b = np.empty((12,4))
all_s = np.empty((12,4))

all_m[:,:]= np.nan
all_t[:,:]= np.nan
all_b[:,:]= np.nan
all_s[:,:]= np.nan

ave1 = ave.copy()
ave1[:,lat>70,:]=np.nan
ave1[:,:,lon<0]=np.nan
ave2 = ave.copy()
ave2[:,lat>70,:]=np.nan
ave2[:,:,lon<-160]=np.nan
ave2[:,:,lon>-90]=np.nan
ave3 = ave.copy()
ave3[:,lat>80,:]=np.nan
ave3[:,:,lon<-60]=np.nan
ave3[:,:,lon>-20]=np.nan

tr1 = trend.copy()
tr1[:,lat>70,:]=np.nan
tr1[:,:,lon<0]=np.nan
tr2 = trend.copy()
tr2[:,lat>70,:]=np.nan
tr2[:,:,lon<-160]=np.nan
tr2[:,:,lon>-90]=np.nan
tr3 = trend.copy()
tr3[:,lat>80,:]=np.nan
tr3[:,:,lon<-60]=np.nan
tr3[:,:,lon>-20]=np.nan


b1 = b.copy()
b1[:,lat>70,:]=np.nan
b1[:,:,lon<0]=np.nan
b2 = b.copy()
b2[:,lat>70,:]=np.nan
b2[:,:,lon<-160]=np.nan
b2[:,:,lon>-90]=np.nan
b3 = b.copy()
b3[:,lat>80,:]=np.nan
b3[:,:,lon<-60]=np.nan
b3[:,:,lon>-20]=np.nan

s1 = std.copy()
s1[:,lat>70,:]=np.nan
s1[:,:,lon<0]=np.nan
s2 = std.copy()
s2[:,lat>70,:]=np.nan
s2[:,:,lon<-160]=np.nan
s2[:,:,lon>-90]=np.nan
s3 = std.copy()
s3[:,lat>80,:]=np.nan
s3[:,:,lon<-60]=np.nan
s3[:,:,lon>-20]=np.nan

for i in range(0,12):
    all_m[i,0] = np.nanmean(ave[i,:,:])
    all_m[i,1] = np.nanmean(ave1[i,:,:])
    all_m[i,2] = np.nanmean(ave2[i,:,:])
    all_m[i,3] = np.nanmean(ave3[i,:,:])
    
    all_s[i,0] = np.nanmean(std[i,:,:])
    all_s[i,1] = np.nanmean(s1[i,:,:])
    all_s[i,2] = np.nanmean(s2[i,:,:])
    all_s[i,3] = np.nanmean(s3[i,:,:])
    
    all_b[i,0] = np.nanmean(b[i,:,:])
    all_b[i,1] = np.nanmean(b1[i,:,:])
    all_b[i,2] = np.nanmean(b2[i,:,:])
    all_b[i,3] = np.nanmean(b3[i,:,:])

    all_t[i,0] = np.nanmean(trend[i,:,:]*10.)
    all_t[i,1] = np.nanmean(tr1[i,:,:]*10.)
    all_t[i,2] = np.nanmean(tr2[i,:,:]*10.)
    all_t[i,3] = np.nanmean(tr3[i,:,:]*10.)
                               
df=pd.DataFrame(all_m)
print(df)
df.to_csv(dirpath + 'all_md_m_gr.csv', mode='w',index=False, header=False)
df=pd.DataFrame(all_t)
print(df)
df.to_csv(dirpath + 'all_md_t_gr.csv', mode='w',index=False, header=False)
df=pd.DataFrame(all_b)
print(df)
df.to_csv(dirpath + 'all_md_b_gr.csv', mode='w',index=False, header=False)
df=pd.DataFrame(all_s)
print(df)
df.to_csv(dirpath + 'all_md_s_gr.csv', mode='w',index=False, header=False)

In [ ]:
cenLon = 0
minLat = 55
maxLat = 90
textLat = minLat - 15
textLon = cenLon
month = ['Jan', 'Feb', 'Mar', 'Apr','May','Jun','Jul','Aug','Sep', 'Oct', 'Nov', 'Dec']

In [ ]:
#Color
contour_levels = np.arange(-5,5.2,0.2)
bwr = cm.get_cmap('bwr', 100)
bwr = bwr(np.arange(100))
#white = np.array([1.,1.,1.,1])
#jet[70:85,:]= white
new_cmap = ListedColormap(bwr)

In [ ]:
stmm = 1
enmm = 12

for mm in np.arange(stmm,enmm+1):

    fig = plt.figure(figsize=(6,6))
    ax = plt.axes(projection=ccrs.NorthPolarStereo(central_longitude = cenLon))
    ax.set_extent([0, 359, 60, 90], crs=ccrs.PlateCarree())

    ax.gridlines(color='gray', linestyle='--',linewidth=0.5)

    # 원형테두리
    theta = np.linspace(0,2*np.pi,144)
    center = [0.5,0.5]
    radius = 0.5
    points = np.array([np.cos(theta),np.sin(theta)]).T
    circle = mpath.Path(points*radius+center)
    ax.set_boundary(circle,transform=ax.transAxes)

    #shading
    image = ax.contourf(lon,lat,trend[mm-1,:,:]*10.,contour_levels, cmap=new_cmap, transform=ccrs.PlateCarree(), extend='both')
    plt.gca().coastlines()
    ax.text(0, 80+0.7, ' 80°N', fontsize = 8.2, ha ='center', color='gray',transform = ccrs.PlateCarree())
    ax.text(0, 70+1.0, ' 70°N', fontsize = 8.2, ha ='center', color='gray',transform = ccrs.PlateCarree())
    ax.text(0, 60+1.3, ' 60°N', fontsize = 8.2, ha ='center', color='gray',transform = ccrs.PlateCarree())
    ax.text(60, 58., ' 60°E', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(120, 58, ' 120°E', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(180, 59.7, ' 180°', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(-60, 57.8, '60°W  ', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(-120, 58.3, '120°W  ', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(0, 58., ' 0°', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())

    #colorbar
#    caxes = fig.add_axes([0.1,0.07,0.86,0.03])
#    cbar =plt.colorbar(image, orientation = 'horizontal', ticks = contour_levels[::10],drawedges = True, cax=caxes)
#    cbar.ax.tick_params(direction='in')
    #cbar.ax.set_xticklables([int(x) for x in contour_levels[::2]])

    #title
#    plt.suptitle('Spatial Interpolation change rate (' +str(month[mm-1]) +')', fontsize = 15, y=0.95)
    #ax.text(textLon, textLat, 'explanation', fontsize = 8, ha ='center', transform = ccrs.Geodetic())

 #    plt.show()
    plt.savefig(dirpath+'trend'+ str(mm) + '_gr.png', dpi=300, trnsparent=True)
    plt.savefig(dirpath+'trend'+ str(mm) + '_gr.eps')

In [ ]:
#Color
contour_levels = np.arange(-56,24,4)
jet = cm.get_cmap('bwr', 100)
jet = jet(np.arange(100))
#white = np.array([1.,1.,1.,1])
#jet[70:85,:]= white
jet = jet[0:68]
new_cmap = ListedColormap(jet)

In [ ]:
stmm = 1
enmm = 12

for mm in np.arange(stmm,enmm+1):

    fig = plt.figure(figsize=(6,6))
    ax = plt.axes(projection=ccrs.NorthPolarStereo(central_longitude = cenLon))
    ax.set_extent([0, 359, 60, 90], crs=ccrs.PlateCarree())

    ax.gridlines(color='gray', linestyle='--',linewidth=0.5)

    # 원형테두리
    theta = np.linspace(0,2*np.pi,144)
    center = [0.5,0.5]
    radius = 0.5
    points = np.array([np.cos(theta),np.sin(theta)]).T
    circle = mpath.Path(points*radius+center)
    ax.set_boundary(circle,transform=ax.transAxes)

    #shading
    image = ax.contourf(lon,lat,ave[mm-1,:,:],contour_levels, cmap=new_cmap, transform=ccrs.PlateCarree(), extend='both')
    plt.gca().coastlines()
    ax.text(0, 80+0.7, ' 80°N', fontsize = 8.2, ha ='center', color='gray',transform = ccrs.PlateCarree())
    ax.text(0, 70+1.0, ' 70°N', fontsize = 8.2, ha ='center', color='gray',transform = ccrs.PlateCarree())
    ax.text(0, 60+1.3, ' 60°N', fontsize = 8.2, ha ='center', color='gray',transform = ccrs.PlateCarree())
    ax.text(60, 58., ' 60°E', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(120, 58, ' 120°E', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(180, 59.7, ' 180°', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(-60, 57.8, '60°W  ', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(-120, 58.3, '120°W  ', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())
    ax.text(0, 58., ' 0°', fontsize = 8.2, ha ='center', color='black',transform = ccrs.PlateCarree())

    #colorbar
#    caxes = fig.add_axes([0.1,0.07,0.86,0.03])
#    cbar =plt.colorbar(image, orientation = 'horizontal', ticks = contour_levels[::2],drawedges = True, cax=caxes)
#    cbar.ax.tick_params(direction='in')
    #cbar.ax.set_xticklables([int(x) for x in contour_levels[::2]])

    #title
#    plt.suptitle('Spatial Interpolation (' +str(month[mm-1]) +')', fontsize = 15, y=0.95)
    #ax.text(textLon, textLat, 'explanation', fontsize = 8, ha ='center', transform = ccrs.Geodetic())

 #    plt.show()
    plt.savefig(dirpath+'mean'+ str(mm) + '_gr.png', dpi=300, trnsparent=True)
    plt.savefig(dirpath+'mean'+ str(mm) + '_gr.eps')

In [ ]:
dirpath = '../data/CRU/grid/'
ncfile = Dataset(dirpath+'cru_ts4.06.1901.2021.tmp.dat.nc')
lat = np.squeeze(ncfile.variables['lat'])
lat = lat[300::]
lon = np.squeeze(ncfile.variables['lon'])

trend=np.load(dirpath +'trend_var.npy')
trend[np.where(trend>100)]=np.nan
trend[np.where(trend<-100)]=np.nan
np.save(dirpath +'trend_var.npy', trend)
trend=np.load(dirpath +'trend_var_gr.npy')
trend[np.where(trend>100)]=np.nan
trend[np.where(trend<-100)]=np.nan
np.save(dirpath +'trend_var_gr.npy',trend)

trend=np.load(dirpath +'trend_var.npy')
all_t = np.empty((12,4))
all_t[:,:]= np.nan

tr1 = trend.copy()
tr1[:,lat>70,:]=np.nan
tr1[:,:,lon<0]=np.nan
tr2 = trend.copy()
tr2[:,lat>70,:]=np.nan
tr2[:,:,lon<-160]=np.nan
tr2[:,:,lon>-90]=np.nan
tr3 = trend.copy()
tr3[:,lat>80,:]=np.nan
tr3[:,:,lon<-60]=np.nan
tr3[:,:,lon>-20]=np.nan



for i in range(0,12):
    all_t[i,0] = np.nanmean(trend[i,:,:]*10.)
    all_t[i,1] = np.nanmean(tr1[i,:,:]*10.)
    all_t[i,2] = np.nanmean(tr2[i,:,:]*10.)
    all_t[i,3] = np.nanmean(tr3[i,:,:]*10.)
                               
df=pd.DataFrame(all_t)
print(df)
df.to_csv(dirpath + 'all_md_tv.csv', mode='w',index=False, header=False)

trend=np.load(dirpath +'trend_var_gr.npy')

all_t = np.empty((12,4))
all_t[:,:]= np.nan

tr1 = trend.copy()
tr1[:,lat>70,:]=np.nan
tr1[:,:,lon<0]=np.nan
tr2 = trend.copy()
tr2[:,lat>70,:]=np.nan
tr2[:,:,lon<-160]=np.nan
tr2[:,:,lon>-90]=np.nan
tr3 = trend.copy()
tr3[:,lat>80,:]=np.nan
tr3[:,:,lon<-60]=np.nan
tr3[:,:,lon>-20]=np.nan

for i in range(0,12):
    all_t[i,0] = np.nanmean(trend[i,:,:]*10.)
    all_t[i,1] = np.nanmean(tr1[i,:,:]*10.)
    all_t[i,2] = np.nanmean(tr2[i,:,:]*10.)
    all_t[i,3] = np.nanmean(tr3[i,:,:]*10.)
                               
df=pd.DataFrame(all_t)
print(df)
df.to_csv(dirpath + 'all_md_tv_gr.csv', mode='w',index=False, header=False)
